# Clasificación de riesgo de salud mental con NLP y embeddings

- Clasificador de urgencia de canalización a salud mental a partir de texto libre, pensado para correr barato en una arquitectura serverless (AWS Lambda).
- El modelo **prioriza** casos para revisión humana, **nunca decide** un diagnóstico ni una intervención clínica.
- Artículo completo, con el diagrama del pipeline y las decisiones de ingeniería: https://fuzzyfrog.ai/es/ai-lab/proyectos/salud/clasificacion-riesgo-salud-mental-nlp-embeddings/
- **Nota:** este notebook entrena con un dataset público de texto sobre salud mental. No contiene datos de ningún paciente, usuario o institución real, ni ninguna credencial de infraestructura.


## Diagrama de arquitectura

- **Entrenamiento (offline):** dataset abierto → preprocesamiento → embeddings preentrenados → clasificador (regresión logística) sobre 7 clases.
- **Producción (online):** respuesta de texto libre del usuario → mismo preprocesamiento y embeddings → clasificador ya entrenado → mapeo a nivel de urgencia (alta/media/baja).
- En la aplicación real, este pipeline vive dentro de una función AWS Lambda ("Lambda de análisis"), detrás de API Gateway, con el resultado guardado en MongoDB Atlas.
- Diagrama editable disponible en el artículo de la plataforma (liga arriba).


## Carga de datos

- Dataset público: [Sentiment Analysis for Mental Health](https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health/data) en Kaggle.
- 52.681 textos etiquetados en 7 categorías: Normal, Depression, Suicidal, Anxiety, Bipolar, Stress, Personality disorder.
- Combina publicaciones de Reddit y Twitter sobre salud mental, curadas a partir de varios datasets públicos.


In [ ]:
import pandas as pd
import numpy as np

# Descarga vía kagglehub (requiere credenciales de Kaggle configuradas)
# import kagglehub
# dataset_path = kagglehub.dataset_download("suchintikasarkar/sentiment-analysis-for-mental-health")

DATASET_PATH = "data/mental_health_statements.csv"  # ajustar a la ruta local tras la descarga

df = pd.read_csv(DATASET_PATH)
df = df.rename(columns={"statement": "texto", "status": "clase"})
df = df.dropna(subset=["texto", "clase"])
print(df.shape)
df.head()


## Explicación de datos

- Cada fila es un texto y su categoría de salud mental asociada.
- El dataset está desbalanceado: Normal, Depression y Suicidal concentran la mayoría de los ejemplos, mientras que Bipolar y Personality disorder tienen muy pocos.
- Para este proyecto, las 7 categorías se agrupan después en 3 niveles de urgencia, mapeo definido por criterio humano, no por el dataset.


In [ ]:
print(df["clase"].value_counts())
print()
print(df["clase"].value_counts(normalize=True).round(3))


## Análisis de datos / EDA

- Antes de entrenar, conviene ver la longitud de los textos y confirmar el desbalance real entre clases.
- Esto ayuda a decidir si hace falta ponderar el entrenamiento por clase.


In [ ]:
df["num_palabras"] = df["texto"].str.split().apply(len)
print(df.groupby("clase")["num_palabras"].describe()[["count", "mean", "50%"]])

# Distribución de clases, para tener presente el desbalance antes de modelar
df["clase"].value_counts().plot(kind="bar", figsize=(8, 4), title="Distribución de clases en el dataset")


## Modelado

- **Representación:** embeddings de un modelo preentrenado ligero tipo MiniLM (`sentence-transformers/all-MiniLM-L6-v2`), en vez de bolsa de palabras clásica.
- **Clasificador:** regresión logística sobre esos embeddings, para las 7 clases originales del dataset.
- Decisión de ingeniería: se descartó ajustar un transformer completo porque el costo de cómputo e inferencia no cabía en una función serverless de nivel gratuito.
- El mapeo final de 7 clases a 3 niveles de urgencia se aplica **después** de la predicción, como una regla de negocio, no como parte del modelo.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer

X_train_texto, X_test_texto, y_train, y_test = train_test_split(
    df["texto"], df["clase"], test_size=0.2, stratify=df["clase"], random_state=42
)

codificador = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

X_train = codificador.encode(X_train_texto.tolist(), show_progress_bar=True, batch_size=64)
X_test = codificador.encode(X_test_texto.tolist(), show_progress_bar=True, batch_size=64)

clasificador = LogisticRegression(max_iter=1000, class_weight="balanced")
clasificador.fit(X_train, y_train)


## Evaluación

- Se reporta exactitud global, pero sobre todo desempeño por clase, porque el dataset está desbalanceado y las clases más urgentes tienen menos ejemplos.
- Después se aplica el mapeo de 7 clases a 3 niveles de urgencia, y se reevalúa el desempeño ya agrupado, que es la métrica que realmente le importa al equipo de psicología.


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = clasificador.predict(X_test)
print("Exactitud global:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred))


In [ ]:
# Mapeo de 7 clases a 3 niveles de urgencia (decisión de criterio humano)
MAPEO_URGENCIA = {
    "Suicidal": "alta",
    "Depression": "media",
    "Anxiety": "media",
    "Bipolar": "media",
    "Normal": "baja",
    "Stress": "baja",
    "Personality disorder": "baja",
}

y_test_urgencia = y_test.map(MAPEO_URGENCIA)
y_pred_urgencia = pd.Series(y_pred, index=y_test.index).map(MAPEO_URGENCIA)

print("Exactitud tras agrupar en 3 niveles de urgencia:",
      round(accuracy_score(y_test_urgencia, y_pred_urgencia), 3))
print()
print(classification_report(y_test_urgencia, y_pred_urgencia))


## Hallazgos principales

- Los embeddings preentrenados capturan matices semánticos que una bolsa de palabras clásica pierde, sin dejar de ser lo bastante ligeros para desplegarse en una función serverless de bajo costo.
- El dataset público está fuertemente desbalanceado, la categoría "Suicidal", la de mayor urgencia clínica, tiene muchos menos ejemplos que "Normal" o "Depression". Ponderar el entrenamiento por clase (`class_weight="balanced"`) ayuda, pero no reemplaza revisar el desempeño por clase antes de confiar en el modelo.
- Agrupar las 7 clases en 3 niveles de urgencia simplifica la interpretación para el equipo de psicología y refleja mejor el objetivo real del sistema, que es priorizar revisión, no diagnosticar.
- El límite más importante del proyecto no fue técnico, fue de diseño: el modelo nunca decide una intervención clínica, solo ordena qué casos revisar primero. Esa decisión se tomó desde el inicio del proyecto, no como ajuste posterior.
